# DatePlannerAgent

基于高德开放平台 REST API 的约会行程规划智能体。

- 数据层：真实 POI 搜索 / 详情 / 距离 / 天气，全部来自高德，缺失字段标「需要确认」
- 兜底：requests → urllib → curl 三后端自动切换 + 重试
- 无需大模型 Key，可直接运行；也可作为 HelloAgents 等 LLM Agent 的工具层

In [ ]:
%pip install -r requirements.txt -q 2>/dev/null || echo '依赖已就绪'

In [ ]:
from date_planner import AMapClient, DatePlanner

# Key 读取顺序：环境变量 AMAP_KEY -> 项目根 .env
client = AMapClient()
print('高德客户端初始化成功')

## 1. 关键词搜索（POI 候选）

In [ ]:
pois = client.text_search('西餐厅', city='北京', offset=5)
for i, p in enumerate(pois, 1):
    biz = p.get('biz_ext', {}) or {}
    print(f"[{i}] {p.get('name')} | 评分:{p.get('rating') or '需要确认'} | 人均:{biz.get('cost') or '需要确认'} | 地址:{p.get('address') or '需要确认'}")

## 2. POI 详情

In [ ]:
if pois:
    detail = client.detail(pois[0]['id'])
    biz = detail.get('biz_ext', {}) or {}
    print('店名:', detail.get('name'))
    print('电话:', detail.get('tel') or '需要确认')
    print('地址:', detail.get('address') or '需要确认')
    print('营业时间:', biz.get('open_time') or '需要确认')
    print('人均:', biz.get('cost') or '需要确认')
    print('评分:', detail.get('rating') or '需要确认')

## 3. 地点间距离（type: 1驾车 / 2骑行 / 3步行）

In [ ]:
r = client.distance('116.397428,39.90923', '116.391275,39.907212', type_='2')
print('天安门 → 故宫（骑行）:', f"{r['km']} km / 约 {r['min']} 分钟" if r else '需要确认')

## 4. 天气

In [ ]:
for c in client.weather('410400'):
    print(c.get('date'), c.get('week'), '|', c.get('dayweather'), '/', c.get('nightweather'),
          '| 白天', c.get('daytemp'), '° 夜间', c.get('nighttemp'), '°', '|', c.get('daywind'), '风', c.get('daypower'))

## 5. 一键演示：需求 → 调研 → 候选

In [ ]:
planner = DatePlanner()
planner.demo(city='北京', keywords='桌游')

## 6. 8 段式报告模板

完整方案生成见 `date_planner/planner.py` 的 `build_report`，
接入 LLM Agent 后即可自动填充：需求总结 / 推荐方向 / 关键事实 / 路线 / 时间交通 / 待确认 / 备用 / 省流版。

参考资料见 `references/`：maps-workflow / dining / outdoor / movie / workshop / performance / exhibition 专项 SOP。

## 7. HelloAgents 框架接入

4 个高德能力已封装为 HelloAgents Tool（`date_planner/hello_tools.py`）：
`amap_text_search` / `amap_detail` / `amap_distance` / `amap_weather`，
注册进 ToolRegistry 后可由 HelloAgentsLLM + ReActAgent 自动调用。

终端体验（无需任何 Key 也能演示注册表）：

```bash
python demo_agent.py --dry-run
```

配置 `.env` 的 LLM 后，由大模型自动完成“需求→调研→方案”：

```bash
python demo_agent.py "帮我在北京找一家评分高的西餐厅，再查一下今天天气"
```

In [ ]:
from date_planner.hello_tools import build_registry

registry = build_registry()          # 未配置 AMAP_KEY 时也可注册，调用时返回友好提示
print('已注册工具:', registry.list_tools())
print(registry.get_tools_description()[:180], '...')